# E791-style $D^+\to\pi^-\pi^+\pi^+$ toy fit

This first tutorial uses the high-level workflow: define the amplitude model, generate pseudo-data directly from it, fit with `FitSession`, and use the automatic report/projection helpers.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    FitSession, ToyBackground, enable_x64, generate_toy,
    plot_dalitz, plot_square_dalitz,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency

enable_x64()


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

rho_x = Parameter.coefficient("rho.x", 1.0, fixed=True, owner="rho")
rho_y = Parameter.coefficient("rho.y", 0.0, fixed=True, owner="rho")
nr_x = Parameter.coefficient("NR.x", 0.35, bounds=(-2.0, 2.0), step=0.02, owner="NR")
nr_y = Parameter.coefficient("NR.y", -0.20, bounds=(-2.0, 2.0), step=0.02, owner="NR")

model = DecayModel(
    channel,
    [
        Resonance("rho", (0, 1), RealImag(rho_x, rho_y),
                  mass=0.7753, width=0.1491, spin=1),
        NonResonant(RealImag(nr_x, nr_y)),
    ],
    normalization_method="square-dalitz",
    normalization_resolution=180,
    normalization_pair=(0, 1),
)

truth = {"rho.x": 1.0, "rho.y": 0.0, "NR.x": 0.35, "NR.y": -0.20}


In [ ]:
data = generate_toy(model, 20_000, parameters=truth, seed=101, pool_size=150_000)
plot_dalitz(data, x="s13", y="s23", title="Generated D+ toy")
plt.show()


In [ ]:
session = FitSession(model, data)
start = {"NR.x": 0.10, "NR.y": 0.10}
result = session.fit(start, simplex=True, ncall=30_000)

session.report(result)
session.plot_projection(result, "s13")
plt.show()
session.plot_projection(result, "s23")
plt.show()
